# 第5日 ノートブック03：YOLO と PatchCore を比べる

# Day 5 Notebook 03: YOLO versus PatchCore

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 2 つの手法の出力の違いを整理できる | 1. Summarise how the two methods differ in output |
| 2. 同じ画像セットで両方の結果を並べられる | 2. Line up both results on the same image set |
| 3. 同じ基準で OK/NG を比べるときの注意を説明できる | 3. Explain the caution needed when comparing OK/NG decisions |
| 4. それぞれの長所・短所を製造の観点で説明できる | 4. Explain strengths and weaknesses from a manufacturing view |
| 5. PVC 検査にどちらを、どの順序で使うか提案できる | 5. Propose which to use and in what order for PVC |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 140 分 / 140 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第4日ノートブック03 と第5日ノートブック02 を実行し、`submissions` に結果が保存されていること。 / You ran Day 4 Notebook 03 and Day 5 Notebook 02, so the results exist in `submissions`. |
| 必要なライブラリ / Required libraries | pandas, numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | `aivision-anomalib`（YOLO の結果はファイルから読み込みます） |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 比較表 | The comparison table |
| 並べた結果画像 | Side-by-side result images |
| PVC 検査への推奨（判断と理由） | A recommendation for PVC with reasons |

> **English:** YOLO and PatchCore live in two different environments, so this notebook compares the **saved results** instead of loading both models. That is a deliberate design choice, documented in `requirements/tested_versions.md`.

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師：2 つの手法の比較の説明 | Instructor: comparing the two approaches |
| 09:00–10:30 | Part 1〜Part 2（出力の違い・結果の読み込み） | Part 1-Part 2 (output types, loading results) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 3〜Part 4（並べて比較・同じ基準で判定） | Part 3-Part 4 (side-by-side, common criterion) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：振り返りと午後の説明 | Instructor: review and briefing |
| 13:50–16:50 | 演習・推奨の作成・提出物 | Exercises, recommendation and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(13, 4.5)):
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

---

## Part 1　出力の違いをおさらいする / Recap: the outputs differ

In [ ]:
show([Image.open(ASSETS / "diagrams" / "diagram_yolo_vs_patchcore.png")],
     ["two different answers to two different questions"], figsize=(11, 5.5))

#### 日本語：同じにできないものは、同じに扱わない
**日本語：** 2 つの手法は、**返すものが違います**。

| | YOLO | PatchCore |
|---|---|---|
| 主な出力 | 四角 + クラス + confidence | 異常スコア + ヒートマップ |
| 「何の欠陥か」 | **分かる** | 分からない |
| 「どこか」 | 四角の位置 | ヒートマップの明るい場所 |
| 学習に要るもの | 欠陥のラベル | 正常品だけ |

**日本語：** したがって、**mAP と AUROC を直接比べてはいけません**。
測っているものが違うからです。

比べてよいのは、**「画像単位で OK/NG を出したときの一致度」** です。
それには **両方にしきい値を決める** 必要があります。

**English:** The two methods **return different things**:

| | YOLO | PatchCore |
|---|---|---|
| main output | box + class + confidence | anomaly score + heatmap |
| "which defect" | **yes** | no |
| "where" | box location | bright regions of the heatmap |
| training needs | defect labels | normal images only |

Therefore **mAP and AUROC must not be compared directly** — they measure different things.

What *can* be compared is **the image-level OK/NG agreement**, and that requires **choosing a
threshold for both**.
#### English: Do not pretend different things are the same

---

## Part 2　保存された結果を読み込む / Loading the saved results

#### 日本語：2 つの JSON を読む
**日本語：** 次の 2 つを読み込みます。

| ファイル | 作った場所 |
|---|---|
| `submissions/day04/yolo_predictions_on_shared_set.json` | 第4日ノートブック03 |
| `submissions/day05/patchcore_predictions.json` | 第5日ノートブック02 |

**日本語：** どちらかが無い場合は、そのノートブックを先に実行してください。
**無い数字を作ってはいけません。**

**English:** Two files are loaded:

| File | Produced by |
|---|---|
| `submissions/day04/yolo_predictions_on_shared_set.json` | Day 4 Notebook 03 |
| `submissions/day05/patchcore_predictions.json` | Day 5 Notebook 02 |

If either is missing, run that notebook first. **Never invent the missing numbers.**
#### English: Read the two JSON files

In [ ]:
yolo_path = COURSE_ROOT / "submissions" / "day04" / "yolo_predictions_on_shared_set.json"
patchcore_path = COURSE_ROOT / "submissions" / "day05" / "patchcore_predictions.json"

missing = [p for p in (yolo_path, patchcore_path) if not p.is_file()]
if missing:
    for path in missing:
        print("見つかりません / missing:", path)
    print()
    print("先に実行してください / run first:")
    if yolo_path in missing:
        print("  notebooks/Day04_YOLO_Training/Day04_03_YOLO_Evaluation.ipynb")
    if patchcore_path in missing:
        print("  notebooks/Day05_PatchCore/Day05_02_PatchCore_Training.ipynb")
else:
    print("両方そろっています / both files found")

In [ ]:
yolo_data = json.loads(yolo_path.read_text(encoding="utf-8")) if yolo_path.is_file() else {"images": []}
patchcore_data = json.loads(patchcore_path.read_text(encoding="utf-8")) if patchcore_path.is_file() else {"images": []}

yolo_frame = pd.DataFrame(yolo_data["images"])
patchcore_frame = pd.DataFrame(patchcore_data["images"])

if len(yolo_frame):
    yolo_frame["image"] = yolo_frame["image"].astype(str)

if len(patchcore_frame):
    # PatchCore 側は "test/xxx/yyy.png" 形式なので、"test/" を外してそろえる
    # PatchCore paths look like "test/xxx/yyy.png"; strip the prefix so they can be joined
    patchcore_frame["image"] = patchcore_frame["image"].str.replace("^test/", "", regex=True)

print("YOLO の行数 / YOLO rows          :", len(yolo_frame))
print("PatchCore の行数 / PatchCore rows:", len(patchcore_frame))
print()
print("YOLO の列 / columns:", list(yolo_frame.columns))
print("PatchCore の列 / columns:", list(patchcore_frame.columns))

---

## Part 3　同じ画像で並べる / Lining them up on the same images

#### 日本語：共通の画像だけで比べる
**日本語：** 2 つの結果は、**同じ画像でしか比べられません**。

また PatchCore は、評価データを検証用と試験用に分けているため、
**全画像の結果は残っていません**。共通部分だけで比べます。

**日本語：** これは弱点ではなく、**正直な比較** です。
無い画像の結果を推測してはいけません。

**English:** The two results can only be compared **on the same images**.

PatchCore also splits its evaluation data into validation and test, so **not every image has a
result**. Compare the intersection only.

That is not a weakness — it is an **honest comparison**. Never guess the missing rows.
#### English: Compare only the images present in both

In [ ]:
if len(yolo_frame) and len(patchcore_frame):
    merged = patchcore_frame.merge(
        yolo_frame,
        on="image",
        how="inner",
        suffixes=("_patchcore", "_yolo"),
    )
    print("共通の画像数 / common images:", len(merged))
    print()
    print(merged.head(8).to_string(index=False))
else:
    merged = pd.DataFrame()
    print("比較できるデータがありません / no data to compare")

---

## Part 4　同じ基準で OK/NG を出す / Making a common OK/NG decision

#### 日本語：しきい値を両方に決める
**日本語：** 画像単位で比べるには、両方を **OK/NG** にする必要があります。

| 手法 | OK/NG の決め方 | この研修の値 |
|---|---|---|
| YOLO | 検出が 1 つ以上あれば NG | `conf >= 0.25` で 1 つ以上 |
| PatchCore | 異常スコアがしきい値以上なら NG | 正常スコア最大と異常スコア最小の間 |

**日本語：** しきい値の決め方は **手法ごとに違う** ので、
「同じ基準で公平に比べた」と言えるのは、**両方とも同じ正解データに対して
しきい値を決めた場合だけ** です。ここでは、その近似として上記の値を使います。

**English:** To compare per image, both must produce OK/NG:

| Method | Decision rule | Value here |
|---|---|---|
| YOLO | NG if at least one detection | at least one with `conf >= 0.25` |
| PatchCore | NG if the score is at or above a threshold | between the max normal and min abnormal score |

The way a threshold is chosen **differs per method**, so "a fair comparison on the same basis" is
only strictly true when both thresholds are set on the same labelled data. The values above are a
practical approximation.
#### English: Choose a threshold for each method

In [ ]:
if len(merged):
    # PatchCore のしきい値: 正常の最大と異常の最小の中央 / midpoint between normal max and abnormal min
    normal_scores = merged.loc[~merged["truly_abnormal"], "score"]
    abnormal_scores = merged.loc[merged["truly_abnormal"], "score"]
    if len(normal_scores) and len(abnormal_scores):
        patchcore_threshold = float((normal_scores.max() + abnormal_scores.min()) / 2)
    else:
        patchcore_threshold = 0.5
    print("PatchCore のしきい値 / threshold:", round(patchcore_threshold, 4))

    yolo_threshold = float(yolo_data.get("conf_threshold", 0.25))
    print("YOLO のしきい値 / threshold    :", yolo_threshold, "(検出が 1 つ以上 / at least one detection)")

    merged["patchcore_ng"] = merged["score"] >= patchcore_threshold
    merged["yolo_ng"] = merged["yolo_detected"].astype(bool)
    merged["truth_ng"] = merged["truly_abnormal"].astype(bool)

    def confusion(column):
        predicted = merged[column]
        truth = merged["truth_ng"]
        return {
            "true_positive": int((predicted & truth).sum()),
            "false_positive": int((predicted & ~truth).sum()),
            "false_negative": int((~predicted & truth).sum()),
            "true_negative": int((~predicted & ~truth).sum()),
        }

    summary = pd.DataFrame({
        "YOLO": confusion("yolo_ng"),
        "PatchCore": confusion("patchcore_ng"),
    })
    print()
    print(summary.to_string())
else:
    summary = pd.DataFrame()

#### 日本語：精度と再現率を計算する
**日本語：** 混同行列から、2 つの指標を計算します。

```text
precision = TP / (TP + FP)      <- 検出したもののうち本物の割合
recall    = TP / (TP + FN)      <- 本物の欠陥のうち捕まえた割合
```

**日本語：** 製造検査では **recall（見逃しの少なさ）** をまず見ます。

**English:** Two metrics follow from the confusion matrix:

```text
precision = TP / (TP + FP)      <- of what we flagged, how much was real
recall    = TP / (TP + FN)      <- of all real defects, how many we caught
```

In inspection, look at **recall (few misses) first**.
#### English: Compute precision and recall

In [ ]:
if len(summary):
    def safe_divide(numerator, denominator):
        return round(numerator / denominator, 3) if denominator else float("nan")

    rows = []
    for method in summary.columns:
        tp = summary.loc["true_positive", method]
        fp = summary.loc["false_positive", method]
        fn = summary.loc["false_negative", method]
        tn = summary.loc["true_negative", method]
        rows.append({
            "method": method,
            "precision": safe_divide(tp, tp + fp),
            "recall": safe_divide(tp, tp + fn),
            "correct": tp + tn,
            "total": tp + fp + fn + tn,
        })
    comparison = pd.DataFrame(rows)
    comparison["accuracy"] = (comparison["correct"] / comparison["total"]).round(3)
    print(comparison.to_string(index=False))

    submission_dir = COURSE_ROOT / "submissions" / "day05"
    comparison.to_csv(submission_dir / "yolo_vs_patchcore.csv", index=False, encoding="utf-8-sig")
    print()
    print("保存 / saved:", submission_dir / "yolo_vs_patchcore.csv")
else:
    comparison = pd.DataFrame()

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
共通の画像数 / common images: 26

PatchCore のしきい値 / threshold: 0.4144
YOLO のしきい値 / threshold    : 0.25 (検出が 1 つ以上 / at least one detection)

                YOLO  PatchCore
true_positive     11         20
false_positive     0          0
false_negative     9          0
true_negative      6          6
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
common images: 26
PatchCore threshold: 0.4144
YOLO threshold: 0.25

                YOLO  PatchCore
true_positive     11         20
false_positive     0          0
false_negative     9          0
true_negative      6          6
```

#### 日本語：この比較の限界
**日本語：** この表を読むときの注意です。**特に 3 番目が重要** です。

1. **画像が少ない** — 数十枚での比較は、偶然で入れ替わります
2. **合成画像** — 実際の PVC ではありません
3. **PatchCore のしきい値はこの比較用データから決めている** — つまり **楽観的** です。
   実務では、**学習にも評価にも使っていない** 正常品でしきい値を決めます。
   したがって、PatchCore 側の数字は **実際より良く出ます**。
4. **PatchCore は種類を言えない** — 同じ土俵ではない部分があります

**日本語：** それでも、**「どちらがどの種類の失敗をするか」** は読み取れます。
そこがこの比較の価値です。「PatchCore の recall が 1.0 だから PatchCore が優れている」と
結論づけてはいけません。

**English:** Cautions when reading the table — **point 3 matters most**:

1. **few images**: with tens of images, results swap by chance
2. **synthetic data**: not real PVC
3. **the PatchCore threshold is derived from this very comparison set**, which makes it
   **optimistic**. In practice the threshold is set on known-good products that were used neither for
   training nor for evaluation. PatchCore's numbers here are therefore **better than reality**.
4. **PatchCore cannot name defects**: some dimensions are not comparable

The table still reveals **which method makes which kind of error**, and that is its value. It does
**not** justify "PatchCore has recall 1.0, therefore PatchCore is better".
#### English: The limits of this comparison

---

## Part 5　結果を並べて見る / Looking at them side by side

In [ ]:
heatmap_dir = COURSE_ROOT / "submissions" / "day05" / "heatmaps"
heatmap_files = sorted(heatmap_dir.glob("*_heatmap.png")) if heatmap_dir.is_dir() else []

print("保存済みヒートマップ / saved heatmaps:", len(heatmap_files))
for path in heatmap_files:
    print("  ", path.name)

if heatmap_files:
    show([Image.open(path) for path in heatmap_files],
         [path.stem.replace("_heatmap", "") for path in heatmap_files],
         figsize=(4.2 * len(heatmap_files), 4.2))
else:
    print()
    print("ヒートマップがありません。第5日ノートブック02 を実行してください。")
    print("No heatmaps; run Day 5 Notebook 02.")

In [ ]:
# 元の画像と並べる / compare with the original images
test_root = ASSETS / "anomaly_dataset" / "pvc_sheet" / "test"
if heatmap_files:
    pairs = []
    titles = []
    for path in heatmap_files[:4]:
        stem = path.stem.replace("_heatmap", "")
        original = next(test_root.rglob(f"{stem}.png"), None)
        if original is None:
            continue
        pairs.append(Image.open(original))
        titles.append(f"original: {stem}")
    if pairs:
        show(pairs, titles, figsize=(4.2 * len(pairs), 4.2))

#### 日本語：気づいてほしいこと
**日本語：** ヒートマップは **「どこが変か」** を示しますが、**「何が変か」** は示しません。

YOLO は **「何が」** を言えますが、**学習した種類しか** 言えません。

**日本語：** この 2 つは **競合ではなく補完** です。

**English:** A heatmap shows **where it looks unusual**, not **what is unusual**.

YOLO can say **what**, but **only for the classes it learned**.

The two are **complementary, not competing**.
#### English: What you should notice

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：比較表を完成させる
**日本語：** 下の表を、あなたの結果で埋めてください。

| 項目 | YOLO | PatchCore |
|---|---|---|
| 学習に必要な画像 | | |
| 学習に必要なラベル | | |
| 出力 | | |
| 欠陥の種類が分かるか | | |
| 位置が分かるか | | |
| 見逃し（FN）の数 | | |
| 誤報（FP）の数 | | |

**English:** Fill in the table with your own results.
#### English: Exercise 1 (basic): complete the comparison table
Done when every cell is filled with a concrete answer or a number from your run.

In [ ]:
# 演習 1 / Exercise 1
rows = [
    ("学習に必要な画像 / training images", "", ""),
    ("学習に必要なラベル / labels", "", ""),
    ("出力 / output", "", ""),
    ("欠陥の種類が分かるか / defect type", "", ""),
    ("位置が分かるか / location", "", ""),
    ("見逃し FN / misses", "", ""),
    ("誤報 FP / false alarms", "", ""),
]
for item, yolo_answer, patchcore_answer in rows:
    print(f"{item:<38} YOLO={yolo_answer or '-':<12} PatchCore={patchcore_answer or '-'}")

#### 日本語：演習 2（中級）：しきい値を動かして比べる
**日本語：** PatchCore のしきい値を 0.3 / 0.5 / 0.8 に変えて、
**見逃しと誤報の数** がどう変わるか表にしてください。
YOLO 側も `conf` を変えて比べられれば、なお良いです。

**English:** Vary the PatchCore threshold over 0.3 / 0.5 / 0.8 and record how the number of misses
and false alarms changes. Comparing with a different YOLO `conf` is a bonus.
#### English: Exercise 2 (intermediate): move the thresholds
Done when three rows exist and the trade-off direction is stated.

In [ ]:
# 演習 2 / Exercise 2
if len(merged):
    for threshold in [0.3, 0.5, 0.8]:
        predicted = merged["score"] >= threshold
        truth = merged["truth_ng"]
        fn = int((~predicted & truth).sum())
        fp = int((predicted & ~truth).sum())
        print(f"threshold={threshold:.1f}  FN={fn}  FP={fp}")

#### 日本語：演習 3（発展）：PVC 検査への推奨を作る
**日本語：** 次の構成で、**あなたの推奨** を書いてください。根拠として、
この 5 日間で得た数字や観察を必ず引用してください。

| 項目 | あなたの推奨 |
|---|---|
| 第 1 段階で使う手法 | |
| その理由 | |
| 第 2 段階で追加する手法 | |
| 見逃しを減らすためにすること | |
| 誤報を減らすためにすること | |
| 第 2 週で確認すべきこと | |

**English:** Write your recommendation in the table above, citing numbers or observations from the
past five days as evidence.
#### English: Exercise 3 (challenge): build a recommendation for PVC
Done when every row is filled and at least two rows quote a concrete result from this course.

In [ ]:
# 演習 3 / Exercise 3
recommendation = {
    "stage1_method": "",
    "reason": "",
    "stage2_method": "",
    "reduce_misses": "",
    "reduce_false_alarms": "",
    "week2_checks": "",
}
for key, value in recommendation.items():
    print(f"{key:<22} {value or '(未記入 / not written)'}")

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. YOLO の mAP と PatchCore の AUROC を直接比べてはいけないのはなぜですか。
2. 画像単位で比べるために、両方に何が必要ですか。
3. PatchCore が見逃しやすいのは、どんな欠陥ですか。
4. YOLO が見逃しやすいのは、どんな欠陥ですか。
5. あなたの現場なら、どちらを先に試しますか。理由も書いてください。

**English:**

1. Why must YOLO's mAP and PatchCore's AUROC not be compared directly?
2. What does each method need in order to be compared at image level?
3. What kind of defect is PatchCore likely to miss?
4. What kind of defect is YOLO likely to miss?
5. On your line, which would you try first, and why?
#### English: Understanding questions
Model answers: `docs/instructor/day05_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `yolo_predictions_on_shared_set.json` が無い | `yolo_predictions_on_shared_set.json` is missing | 第4日ノートブック03 の Part 5 を実行してください。 | Run Part 5 of Day 4 Notebook 03. |
| `patchcore_predictions.json` が無い | `patchcore_predictions.json` is missing | 第5日ノートブック02 の Part 5 を実行してください。 | Run Part 5 of Day 5 Notebook 02. |
| 共通の画像が 0 枚になる | Zero common images | パスの形式が違う可能性があります。`image` 列の先頭 `test/` を外しているか確認してください。 | Path formats may differ. Check that the leading `test/` was stripped. |
| precision や recall が `nan` になる | precision/recall are `nan` | TP + FP または TP + FN が 0 です。異常画像が 1 枚も無い、または 1 枚も検出していません。 | TP + FP or TP + FN is zero: no abnormal images, or nothing detected. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] 2 つの結果ファイルを読み込めた
      - [ ] Loaded both result files
- [ ] 同じ画像で比較表を作った
      - [ ] Built the comparison table on shared images
- [ ] しきい値を動かして FN / FP の変化を見た
      - [ ] Moved thresholds and observed FN/FP
- [ ] 比較の限界を 4 つ説明できる
      - [ ] Can explain the four limits of the comparison
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Anomalib ドキュメント](https://anomalib.readthedocs.io/)
- [Ultralytics 検証モード](https://docs.ultralytics.com/modes/val/)
- [MVTec AD ベンチマーク](https://paperswithcode.com/sota/anomaly-detection-on-mvtec-ad)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** 同じ画像に対して、YOLO で検出した位置と PatchCore のヒートマップの最大点がどれくらい離れているかを、3 枚で測ってみてください。

**English:** For three images, measure how far the YOLO box centre is from the brightest point of the PatchCore heatmap.